<a href="https://colab.research.google.com/github/1kaiser/d_jax/blob/main/jax_3d_reconstruction_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JAX 3D Reconstruction Pipeline (Zonal) 🚀
This notebook provides a high-performance JAX ecosystem for metric 3D reconstruction using **Depth Pro**, **SuperPoint**, and **LightGlue**.

## 1. Environment Setup
Prepare the JAX environment and download model weights.

In [ ]:
# @title 1.1 Setup & Dependencies
import sys
import os

def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

IN_COLAB = is_colab()

if IN_COLAB:
    print("Running in Google Colab")
    os.system("pip install -q jax[tpu] -f https://storage.googleapis.com/jax-releases/libtpu_releases.html")
    os.system("pip install -q flax open3d tqdm opencv-python")
    
    if not os.path.exists("/content/opyf_colab"):
        os.system("git clone https://github.com/1kaiser/opyf_colab.git /content/opyf_colab")
    
    sys.path.append("/content/opyf_colab")
    sys.path.append("/content/opyf_colab/models/jax")
    os.chdir("/content/opyf_colab")
else:
    print("Running Locally")
    project_root = os.getcwd() 
    if project_root not in sys.path:
        sys.path.append(project_root)
    models_path = os.path.join(project_root, "models/jax")
    if models_path not in sys.path:
        sys.path.append(models_path)

import jax
import flax
import open3d as o3d
from tqdm import tqdm
import cv2
print(f"JAX Devices: {jax.devices()}")

In [ ]:
# @title 1.2 Weights Management
import os

def download_weights():
    os.makedirs("weights", exist_ok=True)
    BASE_URL = "https://github.com/1kaiser/d_jax/releases/download/v1.0.0"

    print("Downloading/Verifying weights...")
    
    models = {
        "depth_pro.msgpack": None,
        "superpoint.msgpack": None,
        "superpoint_lightglue.msgpack": None
    }

    for target, parts in models.items():
        target_path = os.path.join("weights", target)
        if os.path.exists(target_path):
            print(f"✅ {target} already exists.")
            continue
            
        print(f"Downloading {target}...")
        url = f"{BASE_URL}/{target}"
        if IN_COLAB:
            os.system(f"wget -q --show-progress {url} -O {target_path}")
        else:
            import subprocess
            subprocess.run(["wget", "-q", "--show-progress", url, "-O", target_path])

    print("\n✅ Required weights ready.")

if IN_COLAB:
    download_weights()
else:
    required = ["depth_pro.msgpack", "superpoint.msgpack", "superpoint_lightglue.msgpack"]
    missing = [r for r in required if not os.path.exists(os.path.join("weights", r))]
    if missing:
        print(f"Missing weights: {missing}. Downloading...")
        download_weights()
    else:
        print("✅ Local weights detected.")

## 2. Standalone Model Testing
Verify individual models before running the full pipeline.

In [ ]:
# @title 2.1 Individual Model Testing
# Detect data path
if IN_COLAB:
    test_img1 = "./data/pinecone_subset/frame_001.jpg"
    test_img2 = "./data/pinecone_subset/frame_002.jpg"
    python_cmd = "python"
else:
    test_img1 = "./data/pinecone_subset/frame_001.jpg"
    test_img2 = "./data/pinecone_subset/frame_002.jpg"
    python_cmd = sys.executable

os.makedirs("output", exist_ok=True)

print(f"Running Depth Pro (Metric Depth) on: {test_img1}")
if IN_COLAB:
    os.system(f"python -m inference.infer_depth_pro --image {test_img1} --weights weights/depth_pro.msgpack --output output/depth_result.jpg")
else:
    import subprocess
    subprocess.run([python_cmd, "-m", "inference.infer_depth_pro", "--image", test_img1, "--weights", "weights/depth_pro.msgpack", "--output", "output/depth_result.jpg"])

print(f"\nRunning LightGlue (Feature Matching) on: {test_img1} and {test_img2}")
if IN_COLAB:
    os.system(f"python -m inference.infer_lightglue --img1 {test_img1} --img2 {test_img2} --output output/match_result.jpg")
else:
    import subprocess
    subprocess.run([python_cmd, "-m", "inference.infer_lightglue", "--img1", test_img1, "--img2", test_img2, "--output", "output/match_result.jpg"])

## 3. Zonal 3D Reconstruction
Run the concentric zonal registration pipeline to align multiple views into a single metric 3D space.

In [ ]:
# @title 3.1 Run Full Zonal Pipeline
from pipelines.pipeline_jax import ReconstructionPipeline

if IN_COLAB:
    image_folder = "./data/pinecone_subset"
else:
    image_folder = "./data/pinecone_subset"

output_dir = "output/pinecone_reconstruction"
os.makedirs(output_dir, exist_ok=True)

pipeline = ReconstructionPipeline(
    "weights/depth_pro.msgpack",
    "weights/superpoint.msgpack",
    "weights/superpoint_lightglue.msgpack"
)

# Radial clip limits the field of view to remove distorted edges
T_globals = pipeline.run(image_folder, output_path=output_dir, num_zones=3, radial_clip=0.7)

print(f"\nChecking for output files in {output_dir}:")
for f in os.listdir(output_dir):
    print(f" - {f}")

# Check if GLB exists
glb_path = os.path.join(output_dir, "mesh_clip0.7.glb")
if os.path.exists(glb_path):
    print(f"\n✅ GLB Mesh generated successfully: {glb_path}")

## 4. Visualization & Export
Interactive visualization and GLB mesh conversion.

In [ ]:
# @title 4.1 Setup Visualization
if IN_COLAB:
    os.system("npm install -g gltf-pipeline")

def view_in_3d(input_path, output_glb="model.glb"):
    """Converts PLY to GLB and renders in-cell."""
    import open3d as o3d
    from IPython.display import HTML, display
    pcd = o3d.io.read_point_cloud(input_path)
    o3d.io.write_point_cloud("temp_cloud.obj", pcd)
    if IN_COLAB:
        os.system(f"gltf-pipeline -i temp_cloud.obj -o {output_glb} --binary")
    else:
        import subprocess
        subprocess.run(["gltf-pipeline", "-i", "temp_cloud.obj", "-o", output_glb, "--binary"])
        
    display(HTML(f"""
    <script type="module" src="https://ajax.googleapis.com/ajax/libs/model-viewer/3.4.0/model-viewer.min.js"></script>
    <model-viewer src="{output_glb}" style="width: 100%; height: 600px;" auto-rotate camera-controls></model-viewer>
    """))


## 5. Citations
```bibtex
@article{sun2026vggt3,
  title={VGG-T³: Offline Feed-Forward 3D Reconstruction at Scale},
  author={Sun, Aljoša and others},
  journal={arXiv preprint arXiv:2602.23361},
  year={2026}
}
```